In [6]:
import cv2
from ultralytics import YOLO
import time
import argparse
import os
import numpy as np

In [7]:
# Define configuration parameters directly
model_path = "D:/aldisetiawan/Semester 5/Computer Vision/car-parking-lot/yolo11l-park-obb.pt"  # Path to YOLOv11 model
video_path = "D:/aldisetiawan/Semester 5/Computer Vision/car-parking-lot/inference/computer_vision_parking_yolo.mp4"  # Path to input video (or use "0" for webcam)
conf_threshold = 0.25  # Confidence threshold
save_results = True  # Whether to save the output video

In [4]:
# # Function to display FPS (Frames Per Second) on the frame
# def show_fps(frame, fps):
#     x, y, w, h = 10, 10, 350, 50  # Define the position and size of the FPS display area
#     cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 0, 0), -1)  # Draw a black rectangle for background
#     cv2.putText(frame, "FPS: " + str(fps), (20, 52), cv2.FONT_HERSHEY_PLAIN, 3.5, (0, 255, 0), 3)  # Add FPS text to the frame

# def show_information(frame, title, counter, box_color):    
#     # Define the size of the box
#     box_width = 250
#     box_height = 50

#     # Get the frame dimensions
#     frame_height, frame_width, _ = frame.shape

#     # Coordinates for the center of the frame
#     center_x = frame_width // 2 
#     center_y = 10  # The y-coordinate position for the top of the box    

#     # Set starting and ending coordinates for the box (centered)
#     start_x = center_x - box_width // 2
#     start_y = center_y
#     end_x = start_x + box_width
#     end_y = start_y + box_height

#     # Draw the box on the frame using cv2.rectangle
#     cv2.rectangle(frame, (start_x, start_y), (end_x, end_y), box_color, -1)  # Filled box

#     # Create the text to display
#     text = f"{title}: {counter}"
#     if(counter == 0):
#         text = f"{title}"
    
#     font = cv2.FONT_HERSHEY_PLAIN
#     font_scale = 2
#     font_thickness = 3

#     text_size = cv2.getTextSize(text, font, font_scale, font_thickness)[0]

#     # Coordinates to center the text inside the box
#     text_x = start_x + (box_width - text_size[0]) // 2
#     text_y = start_y + (box_height + text_size[1]) // 2

#     # Draw the text inside the box using cv2.putText
#     cv2.putText(frame, text, (text_x, text_y), font, font_scale, (0, 0, 0), font_thickness)

# def draw_box(frame, obb, class_id):
#     # Extract the four corner points of the oriented bounding box (OBB)
#     xy1 = obb[0]  # Top-left corner
#     xy2 = obb[1]  # Top-right corner
#     xy3 = obb[2]  # Bottom-right corner
#     xy4 = obb[3]  # Bottom-left corner

#     # Set the default box color to red (BGR format)
#     box_color = (0, 0, 255)  
#     # Change the box color to green if the class ID is 1
#     if(class_id == 1):
#         box_color = (0, 255, 0)

#     # Create a NumPy array representing the four points of the OBB
#     obb_points = np.array([xy1, xy2, xy3, xy4])            
#     # Draw the oriented bounding box on the frame as a closed polyline
#     cv2.polylines(frame, [obb_points], isClosed=True, color=box_color, thickness=2)


In [8]:
def show_fps(frame, fps):
    x, y, w, h = 10, 10, 350, 50
    cv2.rectangle(frame, (x,y), (x+w, y+h), (0, 0, 0), -1)
    cv2.putText(frame, "FPS: " + str(fps), (20, 52), cv2.FONT_HERSHEY_PLAIN, 3.5, (0, 255, 0), 3)

def show_infomation(frame, title, counter, box_color):
    box_width = 250
    box_height = 50
    frame_height, frame_width, _= frame.shape
    center_x = frame_width // 2
    center_y = 10
    start_x = center_x - box_width // 2
    start_y = center_y
    end_x = start_x + box_width
    end_y = start_y + box_height
    cv2.rectangle(frame, (start_x, start_y), (end_x, end_y), box_color, -1)
    text = f"{title}:{counter}" if counter != 0 else f"{title}"
    font = cv2.FONT_HERSHEY_PLAIN
    font_scale = 2
    font_thickness = 3
    text_size = cv2.getTextSize(text, font, font_scale, font_thickness)[0]
    text_x = start_x + (box_width - text_size[0]) //2
    text_y = start_y + (box_height + text_size[1]) // 2
    cv2.putText(frame, text, (text_x, text_y), font, font_scale,(0,0,0), font_thickness)

def draw_box(frame, obb, class_id):
    xy1, xy2, xy3, xy4 = obb
    box_color =(0, 0, 255) if class_id != 1 else (0, 255, 0)
    obb_points = np.array([xy1, xy2, xy3, xy4])
    cv2.polylines(frame, [obb_points], isClosed=True, color=box_color, thickness=2)

In [9]:
if __name__ == '__main__':
    if not os.path.exists(video_path):
        print(f"Video file not found: {video_path}")
        exit()
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Error: Cannot open video file")
        exit()

    output_folder = "result"
    if not os.path.isdir(output_folder):
        os.mkdir(output_folder)

    if save_results:
        filename = os.path.splitext(os.path.basename(video_path))[0]
        output_video_path = f"{output_folder}/{filename}_output.avi"
        frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)

        if fps <=0:
            print("Error: Invalid FPS Value")
            exit()
        
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        writer = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

        model = YOLO(model_path)

        start_time = 0
        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                print("End of video or cannot read the video file.")
                break

            annotated_frame = frame
            results = model(frame, conf=conf_threshold, verbose=False)
            annotated_frame = results[0].plot(line_width=2)

            end_time = time.time()
            fps = 1/(end_time - start_time)
            start_time = end_time

            show_fps(annotated_frame, fps)

            resized_frame = cv2.resize(annotated_frame, (1280, 720))
            cv2.namedWindow("YOLO11 Car Parking Lot Vacancy", cv2.WND_PROP_FULLSCREEN)
            cv2.setWindowProperty("YOLO11 Car Parking Lot Vacancy", cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
            cv2.imshow("YOLO11 Car Parking Lot Vacancy", resized_frame)

            if save_results:
                writer.write(annotated_frame)

            if cv2.waitKey(30) & 0xFF == ord("q"):
                break
        
        if save_results:
            print("The tracking results have been saved in: " + output_video_path)

        cap.release()
        cv2.destroyAllWindows()

The tracking results have been saved in: result/computer_vision_parking_yolo_output.avi


In [ ]:
# if __name__ == '__main__':
#     # Set up video capture
#     video_input = video_path
#     if video_input.isdigit():
#         video_input = int(video_input)
#         cap = cv2.VideoCapture(video_input)  # Open webcam if video_input is a digit
#     else:
#         cap = cv2.VideoCapture(video_input)  # Open video file

#     # Save Video
#     output_folder = "result"  # Directory to save the output video
#     if(not os.path.isdir(output_folder)):  # Create the directory if it doesn't exist
#         os.mkdir(output_folder)

#     if save_results:  # If the save option is selected
#         # Extract the filename from the input video and remove the extension
#         filename = os.path.splitext(os.path.basename(video_path))[0]

#         # Define the path for the output video
#         output_video_path = f"{output_folder}/{filename}_output.mp4"  

#         frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))  # Get frame width
#         frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))  # Get frame height
#         fps = cap.get(cv2.CAP_PROP_FPS)  # Get the frames per second of the input video        

#         # Create video writer objects to save the output video
#         fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Define the codec
#         writer = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))  # Initialize the VideoWriter

#     model = YOLO(model_path)  # Load the YOLO11 model

#     start_time = 0  # Initialize start time for FPS calculation

#     while cap.isOpened():  # Main loop to process video frames
#         success, frame = cap.read()  # Read a frame from the video
#         annotated_frame = frame  # Copy the frame for annotation

#         if success:  # If the frame is read successfully
#             # Perform Instance Segmentation using YOLO
#             results = model(frame, conf=conf_threshold, verbose=False)

#             # Draw the segmentation on the frame
#             annotated_frame = results[0].plot(line_width=2)

#             # Calculate FPS
#             end_time = time.time()  # Get the current time
#             fps = 1 / (end_time - start_time)  # Calculate frames per second
            
#             start_time = end_time  # Update start time for the next frame

#             # Show FPS on the frame
#             show_fps(annotated_frame, fps)  # Call function to display FPS

#             resized_frame = cv2.resize(annotated_frame, (1280, 720))  # Resize frame for display   

#             # Display the annotated frame in a fullscreen window
#             cv2.namedWindow("YOLO11 Car Parking Lot Vacancy", cv2.WND_PROP_FULLSCREEN)  # Create a named window
#             cv2.setWindowProperty("YOLO11 Car Parking Lot Vacancy", cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)  # Set the window to fullscreen
#             cv2.imshow("YOLO11 Car Parking Lot Vacancy", resized_frame)  # Show the annotated frame               

#             if save_results:  # If the save option is selected
#                 writer.write(annotated_frame)  # Write the annotated frame to the output video file               

#             # Break the loop if 'q' is pressed
#             if cv2.waitKey(30) & 0xFF == ord("q"):
#                 break  # Exit loop on 'q' key press
#         else:
#             # Break the loop if the end of the video is reached
#             break

#     if save_results:        
#         print("The tracking results have been saved in: "+ output_video_path)  # Print the save location of the output video


#     # Release the video capture object and close the display window
#     cap.release()  # Release the video capture object
#     cv2.destroyAllWindows()  # Close all OpenCV windows

The tracking results have been saved in: result/parking_output.mp4


In [8]:
video_path = "D:\aldisetiawan\Semester 5\Computer Vision\car-parking-lot\result\parking_output_output.mp4"
cap = cv2.VideoCapture(video_path)

if cap.isOpened():
    print("Error : Could not Open video file")
    exit()
while True:
    ret, frame = cap.read()
    if not ret:
        print("End of Video or connot read the video file")
        break
    
    cv2.imshow("Video", frame)
    if cv2.waitKey(30) & 0xFF == ord("q"):
        break
    
cap.release()
cv2.destroyAllWindows()

End of Video or connot read the video file


In [9]:
print("Width:", cap.get(cv2.CAP_PROP_FRAME_WIDTH))
print("Height:", cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print("FPS:", cap.get(cv2.CAP_PROP_FPS))

Width: 0.0
Height: 0.0
FPS: 0.0
